In [1]:
import sys
sys.path.insert(0, '/Users/victoriajiang/Documents/GitHub/Network-Cyberattack-Detection/verification/svm_t1_poly2_2026-11-27')
sys.path.insert(0, '/Users/victoriajiang/Documents/GitHub/Network-Cyberattack-Detection')
import svm_t1_audit as _svm_t1_audit
_svm_t1_audit.install()

**Linear SVM + T1: degree-2 polynomial features (2,774 columns)**

Use SVM T0's shared splits, LinearSVC settings, six C values and binary metrics,
with LR T1's explicit polynomial transformation. Select C using validation F1,
then evaluate **only that selected model** on the official test split.

The references have no final train-plus-validation refit procedure. We therefore
reuse the selected model fitted on 24,000 training rows; validation remains solely
for selection. The final test result is for this T1 configuration, not a selection
of the best model across the whole project. Do not use it to tune later experiments.

**1. Setup and shared data**

`load_split()` reads the raw split files from `preprocessing.ipynb`; it does not
load an already transformed matrix. Labels and `attack_cat` are not predictors.
Load only training and validation here. Source hashes are recorded for reproducibility.

In [2]:
import sys
sys.path.append("..")

import hashlib
import math
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import Markdown, display
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.preprocessing import DATA, RANDOM_STATE, LOG_COLS, NOT_FEATURES, build_preprocessor, load_split
from src.evaluation import scores, plot_sweep, attack_recall, RESULTS

FIGS = RESULTS.parent / "figures"
FIGS.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
RESULT_CSV = RESULTS / "svm_t1_poly2_results.csv"
FINAL_CSV = RESULTS / "svm_t1_poly2_final_results.csv"
GRID = [1e-3, 1e-2, 1e-1, 1, 10, 100]
BATCH_SIZE = 1024

X_train, y_train, cat_train = load_split("train")
X_val, y_val, cat_val = load_split("val")
train_ids = pd.read_csv(DATA / "processed/train.csv", usecols=["id"])["id"]
val_ids = pd.read_csv(DATA / "processed/val.csv", usecols=["id"])["id"]
if not train_ids.is_unique or not val_ids.is_unique or not set(train_ids).isdisjoint(val_ids):
    raise ValueError("Training/validation source IDs must be unique and disjoint.")
split_hashes = {
    name: hashlib.sha256((DATA / "processed" / f"{name}.csv").read_bytes()).hexdigest()
    for name in ["train", "val"]
}

Check row counts, binary-label mapping, missing/nonfinite values and log-transform
domains. Exact input-vector overlap is audited separately from row-ID overlap;
retaining the existing splits does not guarantee an independent feature holdout.

In [3]:
def check_split(X, y, categories, expected_rows):
    if len(X) != expected_rows or X.shape[1] != 41 or not X.index.equals(y.index):
        raise ValueError("Unexpected split dimensions or target alignment.")
    if set(y.unique()) != {0, 1} or not np.array_equal(y, (categories != "Normal").astype(int)):
        raise ValueError("Binary target labels do not match attack-type metadata.")
    if set(NOT_FEATURES).intersection(X.columns) or X.isna().any().any() or y.isna().any():
        raise ValueError("Missing values or excluded columns found in model inputs.")
    if not np.isfinite(X.select_dtypes(include="number").to_numpy()).all() or (X[LOG_COLS] < 0).any().any():
        raise ValueError("Nonfinite values or negative log-transform inputs.")
    return {"rows": len(X), "features": X.shape[1], "normal": int((y == 0).sum()),
            "attack": int((y == 1).sum()), "attack_share": float(y.mean())}


def overlapping_rows(left, right):
    return len(left.merge(right.drop_duplicates(), on=list(left.columns), how="inner"))


if not X_train.columns.equals(X_val.columns):
    raise ValueError("Training and validation schemas differ.")
data_checks = {
    "train": check_split(X_train, y_train, cat_train, 24_000),
    "validation": check_split(X_val, y_val, cat_val, 6_000),
}
validation_overlap = overlapping_rows(X_val, X_train)
display(pd.DataFrame(data_checks).T.round(4))
print("Train/validation ID overlap: 0; validation rows with a training-feature match:", validation_overlap)

,rows,features,normal,attack,attack_share
train,24000.0,41.0,11559.0,12441.0,0.5184
validation,6000.0,41.0,2890.0,3110.0,0.5183


Train/validation ID overlap: 0; validation rows with a training-feature match: 321


**2. T1 transformation and SVM pipeline**

**Shared preprocessing → polynomial degree 2 → StandardScaler → LinearSVC**.
This is an explicit feature map, not a polynomial-kernel SVC. Both scaling stages
are fitted on training only. All 73 preprocessed features are expanded, including
categorical indicators, preserving LR T1's settings and redundant/zero terms.

$$d=\binom{p+2}{2}-1=\binom{75}{2}-1=2774.$$

The expansion retains 73 linear terms, 73 squares and 2,628 pairwise products.
The second scaler controls their new ranges. The SVM stays linear in these features
but can express quadratic effects in the preprocessed input space. Keep T0's L2,
squared-hinge loss, `dual="auto"`, seed 42 and initial 5,000-iteration limit.
Scores remain decision values with threshold zero, not probabilities.

In [4]:
def build(C, max_iter=5000):
    return make_pipeline(
        build_preprocessor(),
        PolynomialFeatures(degree=2, include_bias=False, interaction_only=False, order="C"),
        StandardScaler(),
        LinearSVC(C=C, dual="auto", max_iter=max_iter, random_state=RANDOM_STATE),
    )

**3. Evaluation helpers**

Batch prediction avoids materializing an approximately 1.7-GiB expanded test
matrix. Every batch uses the already fitted pipeline and is checked for finite
values and the correct feature count. `checked_scores()` uses the project's
attack-positive binary metrics and independently checks them against confusion
counts; FPR is FP divided by the number of normal flows.

Only a convergence warning triggers the same 10,000/20,000-iteration retries as
SVM T0. All warnings and iteration counts are retained. A failed final attempt
is explicitly flagged and excluded from selection; no solver or C is changed.

In [5]:
def predict_batches(pipe, X):
    predictions = []
    for start in range(0, len(X), BATCH_SIZE):
        Z = pipe[:-1].transform(X.iloc[start:start + BATCH_SIZE])
        if Z.shape[1] != pipe[-1].n_features_in_ or not np.isfinite(Z).all():
            raise ValueError("Invalid transformed prediction matrix.")
        predictions.append(pipe[-1].predict(Z))
    return np.concatenate(predictions)


def checked_scores(y, pred, prefix):
    if len(y) != len(pred) or not np.isin(pred, [0, 1]).all():
        raise ValueError("Prediction shape or class labels are invalid.")
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    assert tn + fp + fn + tp == len(y)
    manual = {"precision": tp / (tp + fp) if tp + fp else 0.0,
              "recall": tp / (tp + fn), "f1": 2 * tp / (2 * tp + fp + fn),
              "fpr": fp / (fp + tn)}
    reported = scores(y, pred, prefix)
    for metric, value in manual.items():
        np.testing.assert_allclose(reported[f"{prefix}_{metric}"], value, rtol=0, atol=1e-15)
        assert 0 <= value <= 1
    return reported | {f"{prefix}_{name}": int(value) for name, value in zip(["tn", "fp", "fn", "tp"], [tn, fp, fn, tp])}


def fit_with_retries(C, builder=build):
    total_time, messages = 0.0, []
    for attempt, budget in enumerate([5000, 10000, 20000], start=1):
        pipe = builder(C, max_iter=budget)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            start = time.perf_counter()
            pipe.fit(X_train, y_train)
            fit_time = time.perf_counter() - start
        total_time += fit_time
        converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
        messages.extend(f"{w.category.__name__}: {w.message}" for w in caught)
        for warning in caught:
            print(f"C={C:g}, max_iter={budget}: {warning.category.__name__}: {warning.message}")
        if converged:
            break
    if not converged:
        print(f"C={C:g}: unconverged after {budget} iterations; excluded from selection.")
    return pipe, {"fit_time_s": round(fit_time, 3), "total_fit_time_s": round(total_time, 3),
                  "fit_attempts": attempt, "max_iter_used": budget,
                  "n_iter": int(np.max(pipe[-1].n_iter_)), "converged": converged,
                  "warning_messages": " | ".join(messages)}

**4. Six-C training/validation sweep**

Use T0's C grid from 0.001 (strong regularization) to 100 (weak regularization),
with intermediate values checking the trade-off between fit and shrinkage.
Each candidate gets its own complete pipeline fitted on training only. No
cross-validation is used; if added later, fit this full pipeline inside each fold.
The expected feature count is computed from the fitted transform's actual settings.

In [ ]:
rows, fitted = [], {}
for C in GRID:
    pipe, fit_info = fit_with_retries(C)
    prep, poly, scale, classifier = pipe.steps[0][1], pipe.steps[1][1], pipe.steps[2][1], pipe.steps[3][1]
    base_features = len(prep.get_feature_names_out())
    expected_features = math.comb(base_features + poly.degree, poly.degree) - int(not poly.include_bias)
    assert base_features == 73 and expected_features == poly.n_output_features_ == classifier.n_features_in_ == 2774
    assert len(pipe.steps) == 4 and sum(isinstance(step, PolynomialFeatures) for _, step in pipe.steps) == 1
    assert scale.n_samples_seen_ == len(X_train)
    assert prep.named_transformers_["log"].named_steps["standardscaler"].n_samples_seen_ == len(X_train)
    assert prep.named_transformers_["num"].n_samples_seen_ == len(X_train)
    row = {"model": "SVM", "estimator": "LinearSVC", "transformation": "T1", "param": "C", "value": C, "C": C,
           **fit_info, **checked_scores(y_train, predict_batches(pipe, X_train), "train"),
           **checked_scores(y_val, predict_batches(pipe, X_val), "val")}
    rows.append(row)
    fitted[C] = pipe
    print(f"C={C:g}: train F1={row['train_f1']:.4f}, val F1={row['val_f1']:.4f}, converged={row['converged']}", flush=True)

results = pd.DataFrame(rows)
results["f1_gap"] = results["train_f1"] - results["val_f1"]
results["selected"] = False
metadata = {
    "polynomial_degree": 2, "polynomial_include_bias": False, "polynomial_interaction_only": False,
    "polynomial_order": "C", "post_poly_scaler": "StandardScaler", "post_poly_with_mean": True, "post_poly_with_std": True,
    "preprocessing_order": "shared_preprocessing -> PolynomialFeatures -> StandardScaler -> LinearSVC",
    "random_state": RANDOM_STATE, "penalty": "l2", "loss": "squared_hinge", "dual": "auto", "tol": 1e-4,
    "class_weight": "unweighted", "fit_intercept": True, "intercept_scaling": 1.0, "decision_threshold": 0.0,
    "raw_features": X_train.shape[1], "base_features": base_features, "transformed_features": expected_features,
    "train_size": len(X_train), "validation_size": len(X_val), "positive_label": 1, "metric_average": "binary",
    "selection_metric": "val_f1", "train_val_feature_overlap_rows": validation_overlap,
    "train_split_sha256": split_hashes["train"], "validation_split_sha256": split_hashes["val"],
    "preprocessor_sha256": hashlib.sha256((RESULTS.parent / "src/preprocessing.py").read_bytes()).hexdigest(),
    "sklearn_version": sklearn.__version__, "numpy_version": np.__version__,
}
for name, value in metadata.items():
    results[name] = value
results.to_csv(RESULT_CSV, index=False)
display(results[["C", "train_precision", "train_recall", "train_f1", "train_fpr", "val_precision", "val_recall", "val_f1", "val_fpr", "f1_gap", "fit_time_s", "n_iter", "converged"]].round(4))

**5. Validation curve and selected model**

The shared plot shows train/validation F1 on the left and validation precision,
recall and FPR on the right. A widening gap with worsening validation can suggest
overfitting; polynomial features are not assumed to improve performance.
Select the converged candidate with highest validation F1; exact ties favor the
first, smaller C. This rule and the zero decision threshold are unchanged from T0.

In [ ]:
plot_sweep(results, "SVM T1: polynomial degree 2", FIGS / "curve_svm_t1_poly2.png")
eligible = results[results["converged"]]
if eligible.empty:
    raise RuntimeError("No converged candidate. The CSV is saved; final test evaluation is blocked.")
selected_index = eligible["val_f1"].idxmax()
results.loc[selected_index, "selected"] = True
best = results.loc[selected_index].copy()
best_model = fitted[best["C"]]
assert results["selected"].sum() == 1
results.to_csv(RESULT_CSV, index=False)
print(f"Selected C={best['C']:g}: train F1={best['train_f1']:.4f}, val F1={best['val_f1']:.4f}, "
      f"precision={best['val_precision']:.4f}, recall={best['val_recall']:.4f}, FPR={best['val_fpr']:.4f}")
if best["C"] in [GRID[0], GRID[-1]]:
    print("The winner is at the grid edge; a wider grid is a future experiment, not part of this test.")
if results[["train_f1", "val_f1"]].to_numpy().max() >= 0.999:
    print("Near-perfect scores require scrutiny of split provenance and repeated feature vectors.")
display(Markdown(
    f"Training F1 ranges from {results['train_f1'].min():.4f} to {results['train_f1'].max():.4f}; "
    f"validation F1 ranges from {results['val_f1'].min():.4f} to {results['val_f1'].max():.4f}. "
    f"The largest train-minus-validation gap is {results['f1_gap'].max():.4f}. "
    "These differences describe one validation split; they do not establish statistical significance."
))

The validation confusion matrix uses true labels in rows and predictions in
columns: top-right is false alarms and bottom-left is missed attacks. The
attack-type table uses metadata only and helps identify errors hidden by overall F1.

In [ ]:
pred_val = predict_batches(best_model, X_val)
selected_val_metrics = checked_scores(y_val, pred_val, "val")
for name, value in selected_val_metrics.items():
    np.testing.assert_allclose(best[name], value, rtol=0, atol=1e-15)
ConfusionMatrixDisplay.from_predictions(y_val, pred_val, display_labels=["Normal", "Attack"], cmap="Blues", values_format=",")
plt.title(f"SVM T1, C={best['C']:g}: validation")
plt.savefig(FIGS / "confusion_svm_t1_poly2_validation.png", dpi=150, bbox_inches="tight")
plt.show()
display(attack_recall(y_val, pred_val, cat_val))

**6. Compare with SVM T0 on the same data**

If T0's CSV exists, read its validation-selected C and recheck that single baseline
on the same current training/validation rows and metric definitions. This avoids
assuming an old CSV used identical files. No new T0 search or test evaluation is
performed, and the reference notebook/CSV are unchanged. T1 also adds final
scaling, so differences concern the complete representation rather than products alone.

In [ ]:
def build_t0(C, max_iter=5000):
    return make_pipeline(build_preprocessor(), LinearSVC(C=C, dual="auto", max_iter=max_iter, random_state=RANDOM_STATE))


t0_path = RESULTS / "svm_t0.csv"
if t0_path.is_file():
    recorded_t0 = pd.read_csv(t0_path)
    t0_candidates = recorded_t0[recorded_t0["converged"]]
    if not t0_candidates.empty:
        t0_record = t0_candidates.loc[t0_candidates["val_f1"].idxmax()]
        t0_model, t0_fit = fit_with_retries(t0_record["value"], builder=build_t0)
        if not t0_fit["converged"]:
            raise RuntimeError("T0 comparison did not converge; investigate before interpreting the comparison.")
        t0_metrics = checked_scores(y_train, predict_batches(t0_model, X_train), "train") | checked_scores(y_val, predict_batches(t0_model, X_val), "val")
        display(pd.DataFrame([
            {"representation": "T0 rechecked", "C": t0_record["value"], **t0_metrics},
            {"representation": "T1 selected", "C": best["C"], **{name: best[name] for name in t0_metrics}},
        ])[["representation", "C", "train_f1", "val_precision", "val_recall", "val_f1", "val_fpr"]].round(4))
        largest_t0_difference = max(abs(t0_metrics[name] - t0_record[name]) for name in t0_metrics if name in t0_record)
        print(f"T0 recheck vs recorded metrics: largest difference={largest_t0_difference:.3g}.")
        display(Markdown(f"Selected T1 minus rechecked T0 validation F1: **{best['val_f1'] - t0_metrics['val_f1']:+.4f}**. "
                         "Compare recall and false alarms too; this is not a significance test or a test-set comparison."))
else:
    print("T0 CSV is unavailable; no baseline-performance claim is made.")

**7. Final test of the frozen selected T1 configuration**

Selection is now complete and saved. Load the official test split only here and
reuse the training-only selected pipeline; no scaler, polynomial map or classifier
is refitted on validation/test. Test IDs are local to their source file and may
numerically overlap training IDs. Verify the test's source/order/labels and audit
exact feature matches separately, without removing rows or changing selection.

In [ ]:
frozen_selection = pd.read_csv(RESULT_CSV, keep_default_na=False)
assert frozen_selection["selected"].sum() == 1
assert frozen_selection.loc[frozen_selection["selected"], "C"].iloc[0] == best["C"]
X_test, y_test, cat_test = load_split("test")
if not X_test.columns.equals(X_train.columns):
    raise ValueError("Official test schema differs from training.")
data_checks["test"] = check_split(X_test, y_test, cat_test, 82_332)
processed_test_meta = pd.read_csv(DATA / "processed/test.csv", usecols=["id", "label", "attack_cat"])
official_test_meta = pd.read_csv(DATA / "UNSW_NB15_testing_set.csv", usecols=["id", "label", "attack_cat"])
pd.testing.assert_frame_equal(processed_test_meta, official_test_meta)
assert processed_test_meta["id"].is_unique
print("Official test source, row order and labels verified. IDs are namespaced by source file.")
display(pd.DataFrame(data_checks).T.round(4))
test_train_overlap = overlapping_rows(X_test, X_train)
test_val_overlap = overlapping_rows(X_test, X_val)
test_development_overlap = overlapping_rows(X_test, pd.concat([X_train, X_val], ignore_index=True))
print("Test rows with a training-feature match:", test_train_overlap,
      "; validation-feature match:", test_val_overlap, "; either:", test_development_overlap)

Evaluate the selected model once over the full test set, using batches solely to
limit memory. The final confusion matrix and CSV are final-test results for this
frozen T1 configuration. No other C or T0 model receives test predictions.

In [ ]:
start = time.perf_counter()
pred_test = predict_batches(best_model, X_test)
test_predict_time = time.perf_counter() - start
test_metrics = checked_scores(y_test, pred_test, "test")
assert len(pred_test) == 82_332 and sum(test_metrics[f"test_{name}"] for name in ["tn", "fp", "fn", "tp"]) == len(y_test)
final_record = best.to_dict() | {
    "evaluation_stage": "final_test", "final_refit": False, "model_fit_rows": len(X_train),
    "test_size": len(X_test), "test_evaluations": 1, "test_batch_size": BATCH_SIZE,
    "test_predict_time_s": round(test_predict_time, 3),
    "test_split_sha256": hashlib.sha256((DATA / "processed/test.csv").read_bytes()).hexdigest(),
    "test_feature_overlap_training_rows": test_train_overlap,
    "test_feature_overlap_validation_rows": test_val_overlap,
    "test_feature_overlap_development_rows": test_development_overlap,
    **test_metrics,
}
final_results = pd.DataFrame([final_record])
final_results.to_csv(FINAL_CSV, index=False)
display(pd.DataFrame([
    {"stage": "training", **{metric: best[f"train_{metric}"] for metric in ["precision", "recall", "f1", "fpr"]}},
    {"stage": "selection validation", **{metric: best[f"val_{metric}"] for metric in ["precision", "recall", "f1", "fpr"]}},
    {"stage": "final test", **{metric: test_metrics[f"test_{metric}"] for metric in ["precision", "recall", "f1", "fpr"]}},
]).round(4))
ConfusionMatrixDisplay.from_predictions(y_test, pred_test, display_labels=["Normal", "Attack"], cmap="Blues", values_format=",")
plt.title(f"SVM T1, frozen C={best['C']:g}: final test")
plt.savefig(FIGS / "confusion_svm_t1_poly2_test.png", dpi=150, bbox_inches="tight")
plt.show()
display(Markdown(
    f"Final test F1 is **{test_metrics['test_f1']:.4f}**, versus selection-validation F1 "
    f"**{best['val_f1']:.4f}**. Test false alarms: {test_metrics['test_fp']:,}; "
    f"missed attacks: {test_metrics['test_fn']:,}. "
    "Different capture/class distributions and retained lab shortcuts may contribute "
    "to a validation/test gap; this run alone does not identify the cause. "
    "Test results must not be used to revise C or select another representation."
))

**8. Verify saved CSVs and limitations**

The search CSV has six configurations and exactly one selected row, with no test
metrics. The final CSV has one row with the frozen C and test confusion counts.
Reload both without a dataframe index and compare numerical metrics to the
in-memory results. Binary attack-positive metrics and normal-conditional FPR match T0.

Pre-existing limitations remain: full-file EDA informed preprocessing, repeated
feature vectors can cross source/split boundaries, and TTL may encode lab setup.
The overlap audit reports these cases rather than changing the agreed dataset.
Deployment precision can differ with prevalence. There is no claim that T1 must
outperform T0 or that this selected configuration is the project's global winner.

In [ ]:
saved_search = pd.read_csv(RESULT_CSV, keep_default_na=False)
saved_final = pd.read_csv(FINAL_CSV, keep_default_na=False)
assert len(saved_search) == 6 and saved_search["C"].tolist() == GRID and saved_search["selected"].sum() == 1
assert len(saved_final) == 1 and saved_final.loc[0, "C"] == best["C"] and saved_final.loc[0, "converged"]
assert not any(name.startswith("test_") for name in saved_search.columns)
assert not any(name.startswith("Unnamed:") for name in saved_search.columns) and not any(name.startswith("Unnamed:") for name in saved_final.columns)
metric_columns = [f"{split}_{metric}" for split in ["train", "val"] for metric in ["precision", "recall", "f1", "fpr"]]
np.testing.assert_allclose(saved_search[metric_columns], results[metric_columns], rtol=0, atol=1e-15)
np.testing.assert_allclose(saved_final[[f"test_{metric}" for metric in ["precision", "recall", "f1", "fpr"]]],
                           final_results[[f"test_{metric}" for metric in ["precision", "recall", "f1", "fpr"]]], rtol=0, atol=1e-15)
assert saved_final.loc[0, "test_tn"] + saved_final.loc[0, "test_fp"] + saved_final.loc[0, "test_fn"] + saved_final.loc[0, "test_tp"] == saved_final.loc[0, "test_size"]
print("Verified search CSV:", RESULT_CSV)
print("Verified selected-model final test CSV:", FINAL_CSV)
print("All exported metrics match the in-memory predictions and labels.")

In [ ]:
_svm_t1_audit.capture(globals())